# Construcción de Indicadores Financieros desde Cero (MySQL)

**Proyecto:** ETL + criterio analítico sobre estados contables crudos
**Fuente de datos:** [Financial Data of 4400+ Public Companies](https://www.kaggle.com/datasets/qks1lver/financial-data-of-4400-public-companies) (Kaggle)
**Base de datos:** MySQL (`financial_data`), columnas en español

## Objetivo

Esta versión del proyecto construye los indicadores financieros desde los 3 estados
contables crudos, pero en vez de usar SQLite, se conecta directamente a una base
**MySQL** gestionada en MySQL Workbench, con las columnas ya renombradas a español
(`ticker`, `fecha_cierre`, `ingresos_totales`, `beneficio_neto`...).

**Estructura del notebook:**
1. Conexión a MySQL y carga de los 3 estados
2. Construcción de los 4 bloques de indicadores
3. Tratamiento de valores extremos
4. Ficha financiera final


## 1. Conexión y carga de datos

`indicators_mysql.py` contiene toda la lógica del pipeline. Se conecta a MySQL vía
SQLAlchemy/pymysql, usando las credenciales del bloque `DB_CONFIG` del propio archivo.


In [2]:
!conda install -c conda-forge sqlalchemy pymysql -y

Jupyter detected...
3 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks

  added / updated specs:
    - pymysql
    - sqlalchemy


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.7.22  |       h4c7d964_0         129 KB  conda-forge
    certifi-2026.7.22          |     pyhd8ed1ab_0         134 KB  conda-forge
    greenlet-3.5.4             |  py313h927ade5_0         251 KB  conda-forge
    openssl-3.6.3              |       hf411b9b_0         9.0 MB  conda-forge
    pymysql-1.2.0              |     pyhcf101f3_0          46 KB  conda-forge
    sqlalchemy-2.0.51          |  py313h5fd188c_0         3.6 MB  conda-forge
    -----------------------------------------------------------



==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import sys


sys.path.append('.')

from indicators_mysql import (
    load_joined_data, add_liquidity_ratios, add_profitability_ratios,
    add_solvency_ratios, add_cash_quality_ratios, add_extreme_flags,
    winsorize_ratios, build_financial_indicators
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)


In [2]:
df = load_joined_data()
print(f'Filas tras el join: {len(df)} | Empresas: {df["ticker"].nunique()}')
df.head(3)


Filas tras el join: 17511 | Empresas: 4422


,ticker,fecha_cierre,activo_corriente_total,pasivo_corriente_total,activo_total,pasivo_total,patrimonio_neto_total,efectivo,inventario,cuentas_por_cobrar_netas,deuda_largo_plazo,cuentas_por_pagar,ingresos_totales,beneficio_neto,beneficio_bruto,beneficio_operativo,ebit,coste_ventas,gastos_financieros,flujo_caja_operativo,inversion_capex,dividendos_pagados,endeudamiento_neto
0,A,2016-10-31,3.635000e+09,9.450000e+08,7.794000e+09,3.548000e+09,4.243000e+09,2.289000e+09,533000000.0,714000000.0,1.904000e+09,257000000.0,4202000000,462000000,2197000000,698000000,698000000,2005000000,-72000000.0,7.930000e+08,-139000000.0,-150000000.0,262000000.0
1,A,2017-10-31,4.169000e+09,1.263000e+09,8.426000e+09,3.591000e+09,4.831000e+09,2.678000e+09,575000000.0,801000000.0,1.801000e+09,305000000.0,4472000000,684000000,2399000000,853000000,853000000,2073000000,-79000000.0,8.890000e+08,-176000000.0,-170000000.0,110000000.0
2,A,2018-10-31,3.848000e+09,1.171000e+09,8.541000e+09,3.970000e+09,4.567000e+09,2.247000e+09,638000000.0,835000000.0,1.799000e+09,340000000.0,4914000000,316000000,2680000000,997000000,997000000,2234000000,-75000000.0,1.087000e+09,-177000000.0,-191000000.0,-210000000.0


## 2. Construcción de los indicadores

### 2.1 Liquidez

- **ratio_liquidez** = Activo Corriente / Pasivo Corriente
- **ratio_prueba_acida** = (Activo Corriente − Inventario) / Pasivo Corriente


In [3]:
df = add_liquidity_ratios(df)
df[['ticker', 'fecha_cierre', 'ratio_liquidez', 'ratio_prueba_acida']].describe()


C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,fecha_cierre,ratio_liquidez,ratio_prueba_acida
count,17511,1.734200e+04,1.734200e+04
mean,2018-06-17 04:38:51,inf,inf
min,2012-07-12 00:00:00,0.000000e+00,0.000000e+00
25%,2017-05-31 00:00:00,9.506327e-01,7.356991e-01
50%,2018-04-30 00:00:00,1.735778e+00,1.364388e+00
75%,2019-03-31 00:00:00,3.256246e+00,2.720743e+00
max,2020-05-02 00:00:00,inf,inf
std,NaN,NaN,NaN


### 2.2 Rentabilidad

- **roe** = Beneficio Neto / Patrimonio Neto (⚠️ ver `patrimonio_negativo` si el
  patrimonio es negativo)
- **roa** = Beneficio Neto / Activo Total
- **margen_neto** = Beneficio Neto / Ingresos
- **margen_operativo** = Beneficio Operativo / Ingresos


In [4]:
df = add_profitability_ratios(df)
print(f'Empresas-año con patrimonio neto negativo: {df["patrimonio_negativo"].sum()} '
      f'({df["patrimonio_negativo"].mean()*100:.1f}%)')
df[['ticker', 'fecha_cierre', 'roe', 'roa', 'margen_neto', 'margen_operativo']].head()


Empresas-año con patrimonio neto negativo: 1410 (8.1%)


,ticker,fecha_cierre,roe,roa,margen_neto,margen_operativo
0,A,2016-10-31,0.108885,0.059276,0.109948,0.166111
1,A,2017-10-31,0.141586,0.081177,0.152952,0.190742
2,A,2018-10-31,0.069192,0.036998,0.064306,0.202890
3,A,2019-10-31,0.225569,0.113309,0.207438,0.202014
4,AA,2016-12-31,-0.070746,-0.023893,-0.042928,0.050655


### 2.3 Solvencia

- **deuda_sobre_patrimonio** = Pasivo Total / Patrimonio Neto
- **ratio_endeudamiento** = Pasivo Total / Activo Total
- **cobertura_intereses** = EBIT / Gastos Financieros


In [5]:
df = add_solvency_ratios(df)
df[['ticker', 'fecha_cierre', 'deuda_sobre_patrimonio', 'ratio_endeudamiento',
    'cobertura_intereses']].describe()


C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,fecha_cierre,deuda_sobre_patrimonio,ratio_endeudamiento,cobertura_intereses
count,17511,1.735400e+04,1.735000e+04,1.563900e+04
mean,2018-06-17 04:38:51,2.433500e+04,inf,NaN
min,2012-07-12 00:00:00,-2.361214e+03,0.000000e+00,-inf
25%,2017-05-31 00:00:00,3.836757e-01,3.530631e-01,-4.850244e+00
50%,2018-04-30 00:00:00,1.088204e+00,5.600920e-01,2.595619e+00
75%,2019-03-31 00:00:00,2.732275e+00,7.834661e-01,9.872877e+00
max,2020-05-02 00:00:00,2.445048e+08,inf,inf
std,NaN,2.294479e+06,NaN,NaN


### 2.4 Calidad del beneficio

- **flujo_caja_sobre_beneficio_neto** = Flujo de Caja Operativo / Beneficio Neto
- **flujo_caja_libre** = Flujo de Caja Operativo − CapEx


In [6]:
df = add_cash_quality_ratios(df)
df[['ticker', 'fecha_cierre', 'flujo_caja_sobre_beneficio_neto', 'flujo_caja_libre']].describe()


C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,fecha_cierre,flujo_caja_sobre_beneficio_neto,flujo_caja_libre
count,17511,1.751000e+04,1.751100e+04
mean,2018-06-17 04:38:51,NaN,3.182155e+08
min,2012-07-12 00:00:00,-inf,-1.160380e+11
25%,2017-05-31 00:00:00,2.893727e-01,-1.050893e+07
50%,2018-04-30 00:00:00,1.062276e+00,1.616500e+07
75%,2019-03-31 00:00:00,1.887563e+00,1.522315e+08
max,2020-05-02 00:00:00,inf,7.672600e+10
std,NaN,NaN,2.407415e+09


## 3. Tratamiento de valores extremos

Mismo criterio que en la versión SQLite: el dato individual nunca se altera, solo
se marca (`flag_extremo_<ratio>`). Para agregados/gráficos existe `winsorize_ratios()`.


In [7]:
df = add_extreme_flags(df)
flag_cols = [c for c in df.columns if c.startswith('flag_extremo_')]
print('% de filas marcadas como extremas por ratio:')
print((df[flag_cols].mean() * 100).round(2))


% de filas marcadas como extremas por ratio:
flag_extremo_ratio_liquidez                     1.99
flag_extremo_ratio_prueba_acida                 1.99
flag_extremo_roe                                2.01
flag_extremo_roa                                2.01
flag_extremo_margen_neto                        1.01
flag_extremo_margen_operativo                   1.01
flag_extremo_deuda_sobre_patrimonio             1.99
flag_extremo_ratio_endeudamiento                1.99
flag_extremo_cobertura_intereses                0.00
flag_extremo_flujo_caja_sobre_beneficio_neto    2.01
dtype: float64


C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
C:\Users\JAVI SM\anaconda3\envs\Financialindicatorsusstocks\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


## 4. Pipeline completo y ficha final

`build_financial_indicators()` encapsula todo el proceso anterior en una única
función de entrada, ahora leyendo en vivo desde MySQL.


In [8]:
resultado = build_financial_indicators()
print(f'Ficha financiera generada: {len(resultado)} filas, {resultado["ticker"].nunique()} empresas')
resultado[['ticker', 'fecha_cierre', 'ratio_liquidez', 'roe', 'roa',
           'deuda_sobre_patrimonio', 'flujo_caja_libre']].head(10)


Ficha financiera generada: 17511 filas, 4422 empresas


,ticker,fecha_cierre,ratio_liquidez,roe,roa,deuda_sobre_patrimonio,flujo_caja_libre
0,A,2016-10-31,3.846561,0.108885,0.059276,0.836201,6.540000e+08
1,A,2017-10-31,3.300871,0.141586,0.081177,0.743324,7.130000e+08
2,A,2018-10-31,3.286080,0.069192,0.036998,0.869280,9.100000e+08
3,A,2019-10-31,1.533173,0.225569,0.113309,0.990733,8.660000e+08
4,AA,2016-12-31,1.127614,-0.070746,-0.023893,1.599576,-7.150000e+08
5,AA,2017-12-31,1.303198,0.061685,0.015991,2.354411,8.190000e+08
6,AA,2018-12-31,1.482700,0.044500,0.015497,1.520826,4.900000e+07
7,AA,2019-12-31,1.377292,-0.273589,-0.076892,2.126702,3.070000e+08
8,AAL,2016-12-31,0.744233,0.682695,0.050396,12.546631,7.930000e+08
9,AAL,2017-12-31,0.595520,-1.643590,0.024287,-68.673077,-1.227000e+09


La ficha completa, con formato y gráficos, se exporta a Excel mediante
`build_excel_mysql.py` + `build_charts_mysql.py` →
`ficha_financiera_empresas_mysql.xlsx`.

## Diferencias frente a la versión SQLite

- La fuente de datos es MySQL (Workbench), no un archivo `.db` local
- Los nombres de columna están en español en toda la base y el pipeline
- El mismo pipeline en SQL puro también existe como vistas
  (`v_indicadores_financieros`, `v_indicadores_con_flags`), demostrando que la
  misma lógica de negocio se puede resolver tanto en Python como en SQL puro
